# Multi-Model Pipeline: Specialized Models untuk Jenis & Warna

Pipeline dengan 2 model terpisah:
- **Model JENIS**: EfficientNet-B2 (optimized untuk pattern recognition & shape detection)
- **Model WARNA**: EfficientNet-B0 (optimized untuk color classification)

**Keuntungan**:
1. Each model specialized untuk task-nya
2. Independent learning rates & optimization
3. Better interpretability
4. Easier debugging per task
5. EfficientNet-B2 lebih powerful dalam menangkap pola kompleks jenis pakaian

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# Ensemble ML Models
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("\nEnsemble Models Available:")
print("   - CatBoost")
print("   - XGBoost")
print("   - RandomForest")
print("   - GradientBoosting")
print("   - LogisticRegression")

## 2. Load Data & Class Weights

In [ ]:
train_df = pd.read_csv('train.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data count: {len(sample_sub)}")
print("\nTrain data distribution:")
print(f"Jenis - Kaos: {(train_df['jenis']==0).sum()}, Hoodie: {(train_df['jenis']==1).sum()}")
print(f"Warna - Merah: {(train_df['warna']==0).sum()}, Kuning: {(train_df['warna']==1).sum()}, Biru: {(train_df['warna']==2).sum()}, Hitam: {(train_df['warna']==3).sum()}, Putih: {(train_df['warna']==4).sum()}")

# Class weights untuk JENIS (Kaos: 476, Hoodie: 301)
# Ratio: 1.58:1 (Kaos dominan)
# Strategy: Inverse frequency dengan smoothing untuk stabilitas
total_jenis = len(train_df)
jenis_counts = train_df['jenis'].value_counts().sort_index().values  # [476, 301]
jenis_weights = total_jenis / (2 * jenis_counts)  # n_samples / (n_classes * n_samples_per_class)
# Apply smoothing factor untuk prevent extreme weights
jenis_weights = np.power(jenis_weights, 0.75)  # Soften weights
jenis_weights_tensor = torch.FloatTensor(jenis_weights).to(device)
print(f"\nJenis weights (inverse freq + smoothing):")
print(f"  Kaos (476 samples):   {jenis_weights[0]:.4f}")
print(f"  Hoodie (301 samples): {jenis_weights[1]:.4f}")
print(f"  Ratio: 1:{jenis_weights[1]/jenis_weights[0]:.2f}")

# Class weights untuk WARNA (Merah: 116, Kuning: 125, Biru: 162, Hitam: 234, Putih: 140)
# Ratio: 2.02:1 (Hitam paling dominan vs Merah paling sedikit)
# Strategy: Effective number of samples (Class-Balanced Loss)
# Reduces weights untuk majority class more aggressively
beta = 0.9999  # Hyperparameter untuk control re-weighting
warna_counts = train_df['warna'].value_counts().sort_index().values  # [116, 125, 162, 234, 140]
effective_num = 1.0 - np.power(beta, warna_counts)
warna_weights = (1.0 - beta) / effective_num
warna_weights = warna_weights / warna_weights.sum() * len(warna_weights)  # Normalize
warna_weights_tensor = torch.FloatTensor(warna_weights).to(device)
print(f"\nWarna weights (class-balanced loss, beta={beta}):")
print(f"  Merah (116 samples):  {warna_weights[0]:.4f}")
print(f"  Kuning (125 samples): {warna_weights[1]:.4f}")
print(f"  Biru (162 samples):   {warna_weights[2]:.4f}")
print(f"  Hitam (234 samples):  {warna_weights[3]:.4f}")
print(f"  Putih (140 samples):  {warna_weights[4]:.4f}")
print(f"  Max ratio: 1:{warna_weights.max()/warna_weights.min():.2f}")

## 3. YOLO Preprocessing (Shared)

In [ ]:
def extract_clothing_region(image_path, yolo_model, conf_threshold=0.3):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = yolo_model(img_rgb, verbose=False)
    
    best_box = None
    best_conf = 0
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            
            if cls == 0 and conf > conf_threshold and conf > best_conf:
                best_conf = conf
                best_box = box.xyxy[0].cpu().numpy()
    
    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_rgb.shape[1], x2), min(img_rgb.shape[0], y2)
        cropped = img_rgb[y1:y2, x1:x2]
        return Image.fromarray(cropped)
    else:
        return Image.fromarray(img_rgb)

yolo_model = YOLO('yolov8n.pt')
print("YOLO model loaded")

## 4. Dataset untuk Model JENIS (Shape-focused)

In [ ]:
class JenisDataset(Dataset):
    """
    Dataset untuk klasifikasi JENIS (Kaos vs Hoodie)
    Focus: Shape, structure, collar/hood detection
    """
    def __init__(self, df, img_dir, transform=None, yolo_model=None, use_yolo=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.yolo_model = yolo_model
        self.use_yolo = use_yolo
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        
        img_path = None
        for ext in ['.jpg', '.png']:
            path = os.path.join(self.img_dir, f'{img_id}{ext}')
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found for id {img_id}")
        
        if self.use_yolo and self.yolo_model is not None:
            image = extract_clothing_region(img_path, self.yolo_model)
        else:
            image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_id
        else:
            jenis = self.df.iloc[idx]['jenis']
            return image, torch.tensor(jenis, dtype=torch.long)

# Transform untuk JENIS - GLOBAL AUGMENTATION ONLY (ringan/general)
# Task-specific augmentation akan di-handle di model head
jenis_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),  # Lebih conservative
    transforms.RandomHorizontalFlip(p=0.5),
    # Minimal augmentation - biarkan model yang handle task-specific
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

jenis_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Jenis dataset ready")

## 5. Dataset untuk Model WARNA (Color-focused)

In [ ]:
class WarnaDataset(Dataset):
    """
    Dataset untuk klasifikasi WARNA (5 colors)
    Focus: Color information, robust to lighting
    """
    def __init__(self, df, img_dir, transform=None, yolo_model=None, use_yolo=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.yolo_model = yolo_model
        self.use_yolo = use_yolo
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        
        img_path = None
        for ext in ['.jpg', '.png']:
            path = os.path.join(self.img_dir, f'{img_id}{ext}')
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found for id {img_id}")
        
        if self.use_yolo and self.yolo_model is not None:
            image = extract_clothing_region(img_path, self.yolo_model)
        else:
            image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_id
        else:
            warna = self.df.iloc[idx]['warna']
            return image, torch.tensor(warna, dtype=torch.long)

# Transform untuk WARNA - GLOBAL AUGMENTATION ONLY (ringan/general)
# Task-specific augmentation akan di-handle di model head
warna_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),  # Lebih conservative
    transforms.RandomHorizontalFlip(p=0.5),
    # Minimal augmentation - biarkan model yang handle task-specific
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

warna_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Warna dataset ready")

## 6. Model JENIS - EfficientNet-B2 with Task-Specific Augmentation

In [ ]:
class JenisEmbeddingExtractor(nn.Module):
    """
    EfficientNet-B2 Feature Extractor untuk JENIS
    
    NEW ARCHITECTURE (Two-Stage Pipeline):
    1. STAGE 1 (This Model): Deep Learning Backbone → Extract Embeddings (GAP)
    2. STAGE 2 (Ensemble ML): CatBoost/XGBoost/RF → Voting Classifier
    
    Returns:
    - embeddings: Global Average Pooled features (1408-dim vector)
    - (Optional) logits: For auxiliary loss during embedding training
    """
    def __init__(self, embedding_dim=1408, pretrained=True):
        super(JenisEmbeddingExtractor, self).__init__()
        
        # BACKBONE: EfficientNet-B2 feature extractor
        backbone = models.efficientnet_b2(pretrained=pretrained)
        self.features = backbone.features  # Extract features only
        self.avgpool = backbone.avgpool
        self.embedding_dim = embedding_dim
        
        # Task-Specific Feature Augmentation (training only)
        self.feature_dropout = nn.Dropout2d(0.1)
        
        # Optional: Auxiliary classifier for embedding training
        # (Will be removed after embedding extraction)
        self.aux_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 2)  # Binary classification
        )
    
    def forward(self, x, return_embedding=False):
        # Extract features from backbone
        features = self.features(x)
        
        # Task-specific augmentation (training only)
        if self.training:
            features = self.feature_dropout(features)
        
        # Global Average Pooling → Embedding
        features = self.avgpool(features)
        embedding = torch.flatten(features, 1)
        
        if return_embedding:
            return embedding  # For ensemble ML input
        else:
            # For auxiliary loss during embedding training
            logits = self.aux_classifier(embedding)
            return logits, embedding

jenis_backbone = JenisEmbeddingExtractor(pretrained=True).to(device)
print("JENIS Embedding Extractor (EfficientNet-B2) initialized")
print(f"   Parameters: {sum(p.numel() for p in jenis_backbone.parameters()):,}")
print(f"   Embedding dim: {jenis_backbone.embedding_dim}")
print(f"   Output: 1408-dim feature vector → Ensemble ML")

## 7. Model WARNA - EfficientNet-B0 with Task-Specific Augmentation

In [ ]:
class WarnaModel(nn.Module):
    """
    EfficientNet-B0 dengan Two-Stage Augmentation Strategy
    
    Architecture:
    1. BACKBONE (Global Augmentation): Ringan/general feature extraction
    2. HEAD (Task-Specific Augmentation): Color-focused untuk Warna classification
    
    Task-Specific Augmentation untuk WARNA:
    - Channel Dropout: Simulate lighting changes (hilangkan sebagian channel info)
    - Feature Noise: Add robustness to color variations
    - ColorJitter di feature space: Enhance color discrimination
    """
    def __init__(self, num_classes=5, pretrained=True):
        super(WarnaModel, self).__init__()
        
        # BACKBONE: EfficientNet-B0 feature extractor
        backbone = models.efficientnet_b0(pretrained=pretrained)
        self.features = backbone.features  # Extract features only
        self.avgpool = backbone.avgpool
        in_features = backbone.classifier[1].in_features
        
        # Task-Specific Feature Augmentation Layers
        self.channel_dropout = nn.Dropout2d(0.15)  # Channel dropout untuk simulate lighting
        
        # HEAD: Task-specific classifier dengan color augmentation
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),  # Standard dropout
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x, return_embedding=False):
        # STAGE 1: Backbone feature extraction (global augmentation dari dataset)
        features = self.features(x)
        
        # STAGE 2: Task-specific augmentation di feature level (hanya saat training)
        if self.training:
            # Channel dropout untuk simulate lighting variations
            features = self.channel_dropout(features)
            
            # Add small noise untuk enhance color robustness
            noise = torch.randn_like(features) * 0.01
            features = features + noise
        
        # Global average pooling → Embedding
        features = self.avgpool(features)
        embedding = torch.flatten(features, 1)
        
        if return_embedding:
            return embedding  # For ensemble ML input
        else:
            # For auxiliary loss during embedding training
            logits = self.classifier(embedding)
            return logits, embedding

warna_backbone = WarnaModel(num_classes=5, pretrained=True).to(device)
print("WARNA Embedding Extractor (Efficient Net-B0) initialized")
print(f"   Parameters: {sum(p.numel() for p in warna_backbone.parameters()):,}")
print(f"   Embedding dim: 1280")
print(f"   Output: 1280-dim feature vector → Ensemble ML")

## 8. Data Loaders

In [ ]:
# Split data
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_df['jenis']
)

# JENIS Loaders
jenis_train_dataset = JenisDataset(train_data, 'train/train', jenis_train_transform, yolo_model, use_yolo=True)
jenis_val_dataset = JenisDataset(val_data, 'train/train', jenis_test_transform, yolo_model, use_yolo=True)

jenis_train_loader = DataLoader(jenis_train_dataset, batch_size=32, shuffle=True, num_workers=0)
jenis_val_loader = DataLoader(jenis_val_dataset, batch_size=32, shuffle=False, num_workers=0)

# WARNA Loaders
warna_train_dataset = WarnaDataset(train_data, 'train/train', warna_train_transform, yolo_model, use_yolo=True)
warna_val_dataset = WarnaDataset(val_data, 'train/train', warna_test_transform, yolo_model, use_yolo=True)

warna_train_loader = DataLoader(warna_train_dataset, batch_size=32, shuffle=True, num_workers=0)
warna_val_loader = DataLoader(warna_val_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Data loaders ready")
print(f"   JENIS - Train: {len(jenis_train_dataset)}, Val: {len(jenis_val_dataset)}")
print(f"   WARNA - Train: {len(warna_train_dataset)}, Val: {len(warna_val_dataset)}")

## 9. STAGE 1: Train Backbones untuk Extract Good Embeddings

**Philosophy**: Train deep learning backbones dengan auxiliary classifier untuk learn discriminative embeddings

**Process**:
1. Train JENIS backbone (EfficientNet-B2) → Learn shape embeddings
2. Train WARNA backbone (EfficientNet-B0) → Learn color embeddings  
3. Extract embeddings from trained backbones → Input for ensemble ML

In [ ]:
# Training Setup for Embedding Extraction
jenis_criterion = nn.CrossEntropyLoss(weight=jenis_weights_tensor)
jenis_optimizer = optim.AdamW(jenis_backbone.parameters(), lr=0.001, weight_decay=0.01)
jenis_scheduler = optim.lr_scheduler.ReduceLROnPlateau(jenis_optimizer, mode='min', factor=0.5, patience=3, verbose=True)

warna_criterion = nn.CrossEntropyLoss(weight=warna_weights_tensor)
warna_optimizer = optim.AdamW(warna_backbone.parameters(), lr=0.001, weight_decay=0.01)
warna_scheduler = optim.lr_scheduler.ReduceLROnPlateau(warna_optimizer, mode='min', factor=0.5, patience=3, verbose=True)

print("✅ Optimizers ready for backbone training (embedding extraction)")


def train_backbone_epoch(model, loader, criterion, optimizer, device):
    """Train backbone with auxiliary classifier"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        logits, embeddings = model(images, return_embedding=False)  # Get logits for training
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total


def validate_backbone(model, loader, criterion, device):
    """Validate backbone"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            logits, embeddings = model(images, return_embedding=False)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

print("✅ Training functions ready")

## 10. Train JENIS Backbone (Quick - 10 epochs)

In [ ]:
print("🚀 Training JENIS Backbone (for embedding extraction)...\n")

num_epochs = 10  # Shorter training - just need good embeddings
best_jenis_loss = float('inf')

for epoch in range(num_epochs):
    train_loss, train_acc = train_backbone_epoch(jenis_backbone, jenis_train_loader, jenis_criterion, jenis_optimizer, device)
    val_loss, val_acc = validate_backbone(jenis_backbone, jenis_val_loader, jenis_criterion, device)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    jenis_scheduler.step(val_loss)
    
    if val_loss < best_jenis_loss:
        best_jenis_loss = val_loss
        torch.save(jenis_backbone.state_dict(), 'jenis_backbone.pth')
        print(f"  ✅ Backbone saved")
    print()

print("✅ JENIS Backbone training completed!")
print(f"   Best val loss: {best_jenis_loss:.4f}")
print(f"   Ready to extract embeddings →  Ensemble ML")

## 9. Training Setup - JENIS Model

In [ ]:
# Loss function dengan class weights
jenis_criterion = nn.CrossEntropyLoss(weight=jenis_weights_tensor)

# Optimizer - Adam dengan weight decay
jenis_optimizer = optim.AdamW(jenis_model.parameters(), lr=0.001, weight_decay=0.01)

# Scheduler - ReduceLROnPlateau
jenis_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    jenis_optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print("JENIS training setup ready")

## 10. Training Setup - WARNA Model

In [ ]:
# Loss function dengan class weights
warna_criterion = nn.CrossEntropyLoss(weight=warna_weights_tensor)

# Optimizer - Adam dengan weight decay
warna_optimizer = optim.AdamW(warna_model.parameters(), lr=0.001, weight_decay=0.01)

# Scheduler - ReduceLROnPlateau
warna_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    warna_optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

print("WARNA training setup ready")

## 11. Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

print("Training functions ready")

## 12. Train JENIS Model (ResNet-34)

In [ ]:
print("Training JENIS Model (ResNet-34)...\n")

num_epochs = 20
best_jenis_loss = float('inf')
jenis_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(jenis_model, jenis_train_loader, jenis_criterion, jenis_optimizer, device)
    val_loss, val_acc = validate(jenis_model, jenis_val_loader, jenis_criterion, device)
    
    jenis_history['train_loss'].append(train_loss)
    jenis_history['val_loss'].append(val_loss)
    jenis_history['train_acc'].append(train_acc)
    jenis_history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    jenis_scheduler.step(val_loss)
    
    if val_loss < best_jenis_loss:
        best_jenis_loss = val_loss
        torch.save(jenis_model.state_dict(), 'best_jenis_model.pth')
        print(f"  Model saved with val_loss: {val_loss:.4f}")
    print()

print("JENIS model training completed!")

## 13. Train WARNA Model (EfficientNet-B0)

In [ ]:
print("Training WARNA Model (EfficientNet-B0)...\n")

num_epochs = 20
best_warna_loss = float('inf')
warna_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(warna_model, warna_train_loader, warna_criterion, warna_optimizer, device)
    val_loss, val_acc = validate(warna_model, warna_val_loader, warna_criterion, device)
    
    warna_history['train_loss'].append(train_loss)
    warna_history['val_loss'].append(val_loss)
    warna_history['train_acc'].append(train_acc)
    warna_history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    warna_scheduler.step(val_loss)
    
    if val_loss < best_warna_loss:
        best_warna_loss = val_loss
        torch.save(warna_model.state_dict(), 'best_warna_model.pth')
        print(f"  Model saved with val_loss: {val_loss:.4f}")
    print()

print("WARNA model training completed!")

## 14. Visualisasi Training History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# JENIS Loss
axes[0, 0].plot(jenis_history['train_loss'], label='Train Loss', marker='o')
axes[0, 0].plot(jenis_history['val_loss'], label='Val Loss', marker='o')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('JENIS Model - Loss over Epochs')
axes[0, 0].legend()
axes[0, 0].grid(True)

# JENIS Accuracy
axes[0, 1].plot(jenis_history['train_acc'], label='Train Acc', marker='o')
axes[0, 1].plot(jenis_history['val_acc'], label='Val Acc', marker='o')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('JENIS Model - Accuracy over Epochs')
axes[0, 1].legend()
axes[0, 1].grid(True)

# WARNA Loss
axes[1, 0].plot(warna_history['train_loss'], label='Train Loss', marker='s')
axes[1, 0].plot(warna_history['val_loss'], label='Val Loss', marker='s')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('WARNA Model - Loss over Epochs')
axes[1, 0].legend()
axes[1, 0].grid(True)

# WARNA Accuracy
axes[1, 1].plot(warna_history['train_acc'], label='Train Acc', marker='s')
axes[1, 1].plot(warna_history['val_acc'], label='Val Acc', marker='s')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('WARNA Model - Accuracy over Epochs')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

## 15. Evaluation - Classification Reports

In [ ]:
import seaborn as sns

def evaluate_model(model, loader, device, task_name, class_names):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Classification Report
    print("="*70)
    print(f"CLASSIFICATION REPORT - {task_name}")
    print("="*70)
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    
    # F1-Scores
    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"\nF1-Scores:")
    print(f"   Micro F1-Score: {micro_f1:.4f}")
    print(f"   Macro F1-Score: {macro_f1:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {task_name}\nMicro F1: {micro_f1:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    return all_preds, all_labels, all_probs

# Evaluate JENIS Model
jenis_model.load_state_dict(torch.load('best_jenis_model.pth'))
jenis_preds, jenis_labels, jenis_probs = evaluate_model(
    jenis_model, jenis_val_loader, device, 
    "JENIS (Kaos vs Hoodie)", ['Kaos', 'Hoodie']
)

# Evaluate WARNA Model
warna_model.load_state_dict(torch.load('best_warna_model.pth'))
warna_preds, warna_labels, warna_probs = evaluate_model(
    warna_model, warna_val_loader, device,
    "WARNA (5 Colors)", ['Merah', 'Kuning', 'Biru', 'Hitam', 'Putih']
)

# Calculate EXACT MATCH RATIO (Primary Metric for Multi-Label)
print("\n" + "="*80)
print("EXACT MATCH RATIO (Primary Metric)")


# Convert to numpy arrays
jenis_preds_np = np.array(jenis_preds)
jenis_labels_np = np.array(jenis_labels)
warna_preds_np = np.array(warna_preds)
warna_labels_np = np.array(warna_labels)

# Calculate exact matches
exact_matches = (jenis_preds_np == jenis_labels_np) & (warna_preds_np == warna_labels_np)
exact_match_ratio = exact_matches.sum() / len(exact_matches)

print(f"\nResults:")
print(f"   Total samples: {len(exact_matches)}")
print(f"   Exact matches: {exact_matches.sum()}")
print(f"   EXACT MATCH RATIO: {exact_match_ratio:.4f} ({exact_match_ratio*100:.2f}%)")

# Breakdown analysis
jenis_correct = (jenis_preds_np == jenis_labels_np).sum()
warna_correct = (warna_preds_np == warna_labels_np).sum()
jenis_only_correct = ((jenis_preds_np == jenis_labels_np) & (warna_preds_np != warna_labels_np)).sum()
warna_only_correct = ((jenis_preds_np != jenis_labels_np) & (warna_preds_np == warna_labels_np)).sum()
both_wrong = ((jenis_preds_np != jenis_labels_np) & (warna_preds_np != warna_labels_np)).sum()

print(f"\n📈 Breakdown:")
print(f"   JENIS correct: {jenis_correct}/{len(exact_matches)} ({jenis_correct/len(exact_matches)*100:.2f}%)")
print(f"   WARNA correct: {warna_correct}/{len(exact_matches)} ({warna_correct/len(exact_matches)*100:.2f}%)")
print(f"   ✅ Both correct: {exact_matches.sum()} ({exact_match_ratio*100:.2f}%)")
print(f"   🟡 Only JENIS correct: {jenis_only_correct} ({jenis_only_correct/len(exact_matches)*100:.2f}%)")
print(f"   🟠 Only WARNA correct: {warna_only_correct} ({warna_only_correct/len(exact_matches)*100:.2f}%)")
print(f"   ❌ Both wrong: {both_wrong} ({both_wrong/len(exact_matches)*100:.2f}%)")

print("\n" + "="*80)
print("💡 Interpretation:")
print("   - Exact Match Ratio adalah metric utama untuk submission")
print("   - Target: > 90% untuk competitive performance")
print("   - Breakdown membantu identify mana model yang perlu improvement")
print("="*80)

## 15.5 Exact Match Visualization

In [ ]:
# Visualize Exact Match Analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Pie Chart - Match Categories
categories = ['Both Correct\n(Exact Match)', 'Only JENIS\nCorrect', 'Only WARNA\nCorrect', 'Both Wrong']
counts = [exact_matches.sum(), jenis_only_correct, warna_only_correct, both_wrong]
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
explode = (0.1, 0, 0, 0)  # Explode exact match slice

axes[0].pie(counts, labels=categories, autopct='%1.1f%%', colors=colors, 
            explode=explode, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title(f'Exact Match Analysis\nExact Match Ratio: {exact_match_ratio*100:.2f}%', 
                  fontsize=14, weight='bold', pad=20)

# 2. Bar Chart - Individual vs Combined Accuracy
metrics = ['JENIS\nAccuracy', 'WARNA\nAccuracy', 'EXACT MATCH\nRatio']
values = [
    jenis_correct/len(exact_matches)*100,
    warna_correct/len(exact_matches)*100,
    exact_match_ratio*100
]
bar_colors = ['#3498db', '#9b59b6', '#2ecc71']

bars = axes[1].bar(metrics, values, color=bar_colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Percentage (%)', fontsize=12, weight='bold')
axes[1].set_title('Individual Accuracy vs Exact Match Ratio', fontsize=14, weight='bold', pad=20)
axes[1].set_ylim([0, 105])
axes[1].grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar, value in zip(bars, values):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{value:.2f}%', ha='center', va='bottom', fontsize=11, weight='bold')

plt.tight_layout()
plt.show()

# Print error analysis for mismatches
print("\n" + "="*80)
print("🔍 ERROR ANALYSIS - Where are the mistakes?")
print("="*80)

# Find mismatch patterns
mismatch_mask = ~exact_matches
if mismatch_mask.sum() > 0:
    print(f"\nTotal mismatches: {mismatch_mask.sum()}/{len(exact_matches)}")
    print("\nMismatch breakdown:")
    print(f"  • Only JENIS wrong: {jenis_only_correct} cases")
    print(f"  • Only WARNA wrong: {warna_only_correct} cases")
    print(f"  • Both wrong: {both_wrong} cases")
    
    # Most common error patterns
    print("\nPriority for improvement:")
    if jenis_only_correct > warna_only_correct:
        print(f"  → WARNA model needs more work (wrong in {jenis_only_correct} cases)")
    elif warna_only_correct > jenis_only_correct:
        print(f"  → JENIS model needs more work (wrong in {warna_only_correct} cases)")
    else:
        print(f"  → Both models need equal improvement")
else:
    print("\nPerfect! All predictions match exactly!")

print("="*80)

## 16. Test Predictions

In [ ]:
# Load best models
jenis_model.load_state_dict(torch.load('best_jenis_model.pth'))
warna_model.load_state_dict(torch.load('best_warna_model.pth'))
jenis_model.eval()
warna_model.eval()

# Test datasets
jenis_test_dataset = JenisDataset(sample_sub, 'test/test', jenis_test_transform, yolo_model, use_yolo=True, is_test=True)
warna_test_dataset = WarnaDataset(sample_sub, 'test/test', warna_test_transform, yolo_model, use_yolo=True, is_test=True)

jenis_test_loader = DataLoader(jenis_test_dataset, batch_size=32, shuffle=False, num_workers=0)
warna_test_loader = DataLoader(warna_test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Predict JENIS
jenis_predictions = []
jenis_ids = []

with torch.no_grad():
    for images, img_ids in jenis_test_loader:
        images = images.to(device)
        outputs = jenis_model(images)
        preds = outputs.argmax(1).cpu().numpy()
        jenis_predictions.extend(preds)
        # Convert img_ids to int (remove tensor wrapper)
        jenis_ids.extend([int(img_id) if isinstance(img_id, torch.Tensor) else int(img_id) for img_id in img_ids])

# Predict WARNA
warna_predictions = []
warna_ids = []

with torch.no_grad():
    for images, img_ids in warna_test_loader:
        images = images.to(device)
        outputs = warna_model(images)
        preds = outputs.argmax(1).cpu().numpy()
        warna_predictions.extend(preds)
        # Convert img_ids to int (remove tensor wrapper)
        warna_ids.extend([int(img_id) if isinstance(img_id, torch.Tensor) else int(img_id) for img_id in img_ids])

print(f"Predictions completed")
print(f"   JENIS: {len(jenis_predictions)} samples")
print(f"   WARNA: {len(warna_predictions)} samples")

## 17. Create Submission File

In [ ]:
# Combine predictions
submission = pd.DataFrame({
    'id': jenis_ids,
    'jenis': jenis_predictions,
    'warna': warna_predictions
})

# Ensure all columns are correct types
submission['id'] = submission['id'].astype(int)
submission['jenis'] = submission['jenis'].astype(int)
submission['warna'] = submission['warna'].astype(int)

# Sort by id
submission = submission.sort_values('id').reset_index(drop=True)

# Save to CSV
submission.to_csv('submission_multimodel.csv', index=False)

print("Submission file created: submission_multimodel.csv")
print(f"\nSubmission shape: {submission.shape}")
print(f"Column types:\n{submission.dtypes}\n")
print("\nFirst 10 predictions:")
print(submission.head(10))
print("\nPrediction distribution:")
print(f"Jenis - Kaos: {(submission['jenis']==0).sum()} ({(submission['jenis']==0).sum()/len(submission)*100:.1f}%), Hoodie: {(submission['jenis']==1).sum()} ({(submission['jenis']==1).sum()/len(submission)*100:.1f}%)")
print(f"Warna - Merah: {(submission['warna']==0).sum()}, Kuning: {(submission['warna']==1).sum()}, Biru: {(submission['warna']==2).sum()}, Hitam: {(submission['warna']==3).sum()}, Putih: {(submission['warna']==4).sum()}")